In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess
from string import Template
from IPython.display import Markdown, display


with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils
import llm_tools as llt

from matplotlib import pyplot as plt


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000
REPLACE = False

In [3]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)


In [4]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [5]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The agenda items have been grouped into three clusters. Several examples are provided from each cluster. Summarize the typical kind of case each cluster contains, and highlight the main differences between the clusters.
                  
$examples
""")

examples_text = ""
for cluster in [0, 1, 2]:
    mask = df['cluster']==cluster
    rows = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    examples_text += "------------------------\n"
    examples_text += f"Cluster: {cluster}\n\n"
    for i, row in rows.iterrows():
        examples_text += f"Example {i+1}:\n"
        examples_text += f"{row['agenda_summary']}\n\n"

prompt = PROMPT.substitute(examples=examples_text)

response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)

display(Markdown(response['response']))


## Cluster Summaries and Differences

### Cluster 0: Appeals of Tract Map Approvals for Large-Scale Developments

These cases involve **appeals of Advisory Agency decisions on Vesting Tentative Tract Maps (VTTs)**. The projects are typically large-scale developments—often mixed-use—that require the **merger and resubdivision of land into ground lots and airspace lots** (for condominium purposes). A recurring feature is the inclusion of **haul routes for the export of large quantities of soil** (often tens or hundreds of thousands of cubic yards). Many cases also involve **certification of Environmental Impact Reports (EIRs)** and adoption of environmental findings and mitigation measures. The appellants challenge the Advisory Agency's approval of the tract map and/or the associated CEQA determinations. These projects tend to be substantial in scale (multi-acre sites, hundreds of units, commercial components) and are concentrated in downtown and other major plan areas.

### Cluster 1: Citywide Code Amendments, Ordinances, and Major Institutional/Entitlement Projects

This cluster encompasses two related types of cases. The first and most prominent type involves **citywide or area-wide legislative actions**: proposed ordinances amending the Los Angeles Municipal Code, specific plan amendments, and new regulatory programs (e.g., the Citywide Housing Incentive Program, Mello Act implementation, emergency shelter code alignment, technical corrections to the Processes and Procedures Ordinance). These are **policy-level, legislative items** where the Commission makes recommendations to City Council. The second type involves **major institutional projects** requiring multiple discretionary entitlements—such as zone changes, conditional use permits, coastal development permits, and variances—for facilities like hospitals and schools. What unites the cluster is the **regulatory, procedural, or policy-driven nature** of the actions, as opposed to appeals of individual residential or tract map decisions.

### Cluster 2: Site-Specific Residential and Mixed-Use Development Projects (Often with Density Bonuses/TOC Incentives or TFARs)

These cases involve **individual development projects seeking discretionary approvals**, most commonly for **multi-family residential or mixed-use buildings**. A dominant theme is the use of **affordable housing incentive programs**—particularly **Transit Oriented Communities (TOC)** incentives and **Density Bonus** provisions—to obtain increases in density, height, and FAR, along with reductions in setbacks, open space, and parking requirements, in exchange for setting aside units for low-income households. Several cases are **appeals of the Director of Planning's approval** of such projects. The cluster also includes **high-rise downtown projects** utilizing **Transfers of Floor Area Rights (TFAR)**, Master Conditional Use Permits, and Site Plan Reviews. The common thread is that these are **project-level entitlement decisions** focused on the design, density, and zoning compliance of specific buildings.

---

### Key Differences

| Dimension | Cluster 0 | Cluster 1 | Cluster 2 |
|---|---|---|---|
| **Primary action** | Appeal of tract map (VTT) approval | Code amendments, ordinances, and major institutional entitlements | Site-specific project entitlements (density bonus, TOC, TFAR, site plan review) |
| **Scale/scope** | Large multi-acre developments with subdivision of land | Citywide/area-wide policy or large institutional facilities | Individual building projects (mid-rise to high-rise) |
| **Central issue** | Land subdivision, lot mergers, airspace lots, haul routes | Legislative/regulatory changes or complex multi-entitlement institutional projects | Density, height, and zoning incentives tied to affordable housing commitments |
| **CEQA focus** | Often involves EIR certification challenges | Exemptions, negative declarations, or addenda to prior EIRs | Typically categorical exemptions (Class 32) or prior EIR findings |
| **Procedural posture** | Appeals of Advisory Agency decisions | Commission recommendations to City Council (legislative) or initial entitlement approvals | Appeals of Director of Planning decisions or initial Commission review |
| **Affordable housing nexus** | Not a central feature | May appear in policy context (e.g., CHIP, Mello Act) | Central feature—TOC tiers, density bonus set-asides drive project design |

In [6]:
text = r"""Cluster 0, the smallest cluster, consists mainly of non-development actions, such as citywide code amendments, plan updates, or conditional use permits on existing properties. 
Cluster 1 consists primarily of large 
scale private development projects in which the CPC is the initial approver. 
Cluster 2 consists primarily of appeals of decisions made by lower level agencies."""

with open(os.path.join(LOCAL_PATH, "results/cluster_summaries.tex"), 'w') as f:
    f.write(text)

In [7]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The first example has been identified via semantic embeddings as being especially unique or an outlier relative to the other examples. Please describe what makes this example unique compared to the others.
                  
$examples
""")

output_text = ""
for cluster in [0, 1, 2]:
    output_text += "------------------------\n"
    output_text += f"# Cluster: {cluster}\n\n"
    mask = df['cluster']==cluster
    regular_examples = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    mask2 = (df['cluster']==cluster) 
    unique_example = df.loc[mask2].sort_values(by='mahalanobis', ascending=False).head(1).reset_index(drop=True).iloc[0]
    examples_text = f"## Unique Example:\n\n"
    examples_text += f"{unique_example['agenda_summary']}\n\n"
    output_text += f"## Unique Example:\n\n"
    output_text += f"{unique_example['date']} - {unique_example['item_no']}\n\n" 
    output_text += f"{unique_example['agenda_summary']}\n\n"
    for i, row in regular_examples.iterrows():
        examples_text += f"## Regular Example {i+1}:\n\n"
        examples_text += f"{row['agenda_summary']}\n\n"
    prompt = PROMPT.substitute(examples=examples_text)
    response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    output_text += "# LLM Response:\n\n"
    output_text += f"{response['response']}\n\n"

display(Markdown(output_text))


------------------------
# Cluster: 0

## Unique Example:

2018-12-13 - 7

Appeal of the Deputy Advisory Agency's approval of a Vesting Tentative Tract Map (VTT-74200) on a 4.9-acre site at 129-135 West College Street and 924 North Spring Street in the Central City North Plan Area. The project involves street vacation purposes. The appeal challenges both the certification of the College Station Project EIR and the approval of the Vesting Tentative Tract Map. The applicant is Jeffrey Goldberger of Atlas Capital Group LLC, and the appellants are labor and community organizations.

# LLM Response:

## What Makes the Unique Example Distinctive

The most notable distinguishing feature of the unique example is that the project involves **street vacation purposes** rather than the typical subdivision purposes seen in all the regular examples. Every regular example involves subdivision for residential, commercial, or mixed-use development — creating condominium units, small lots, airspace lots, or ground lots to facilitate construction of housing, commercial space, or live/work units. The unique example's tract map, by contrast, is tied to **street vacation**, which is a fundamentally different land use action involving the relinquishment of public right-of-way.

Several other contrasts reinforce its outlier status:

1. **Absence of detailed development program**: All regular examples describe specific building programs — unit counts, square footages, parking spaces, building heights, income-restricted units, etc. The unique example provides **no such development specifics**, making it unusually sparse in describing what would actually be built.

2. **No subdivision configuration described**: The regular examples consistently detail the lot reconfiguration (e.g., "one master lot and three airspace lots," "12 small lots," "one ground lot and four airspace lots"). The unique example mentions no such subdivision structure — only "street vacation purposes."

3. **Appellants are labor AND community organizations**: While several regular examples feature labor unions (e.g., Carpenters, Laborers' Local 300) or community groups (Hollywood Coalition) as appellants, the unique example specifically identifies **both labor and community organizations** together, suggesting a broader coalition of opposition.

4. **Project naming**: The EIR is referred to as the "College Station Project EIR," giving it a branded project name, which is uncommon among the other examples where EIRs are referenced generically.

In essence, the unique example stands apart because it centers on a **street vacation** rather than a conventional development subdivision, lacks the granular project details present in every other example, and suggests a qualitatively different type of land use action being contested.

------------------------
# Cluster: 1

## Unique Example:

2020-08-13 - 5a

The applicant, Hollywood Forever, Inc., seeks approval for the continued operations of an existing 53-acre cemetery in Hollywood, including up to 150 annual special, community, and cultural events (such as charitable fundraisers, film screenings, concerts, and cultural festivals) with alcohol service permitted between 11:00 a.m. and 2:00 a.m. daily. No new permanent structures are proposed. Requested actions include a CEQA categorical exemption, a Conditional Use to allow ancillary special events (up to 150 per year) in the A1 Zone, and a Conditional Use to permit the sale and dispensing of full-line alcoholic beverages for on-site consumption under a Type 88 ABC license.

# LLM Response:

## What Makes the Unique Example an Outlier

The Hollywood Forever Cemetery item stands out from the other examples in several fundamental ways:

### 1. **An Existing Cemetery as the Subject Property**
This is the only example involving a cemetery — a land use that is inherently unusual in urban planning contexts. Cemeteries are rarely the subject of planning commission actions because they are typically static, long-established uses. None of the other examples involve anything remotely similar; they deal with community plan updates, studio expansions, residential subdivisions, storage facilities, schools, and signage.

### 2. **Entertainment and Event Programming as the Core Request**
The central ask is permission to hold **up to 150 special events per year** — film screenings, concerts, cultural festivals, and charitable fundraisers — at a cemetery. This juxtaposition of a traditionally solemn, passive land use with active, high-frequency entertainment programming is highly unusual. The other examples involve conventional development activities: constructing buildings, changing zoning, modifying setbacks, or expanding enrollment.

### 3. **Alcohol Service at a Cemetery**
The request for a **Type 88 ABC license** (full-line alcoholic beverages) for on-site consumption at a cemetery, with service hours extending until 2:00 a.m., is strikingly atypical. No other example involves alcohol licensing, and the combination of alcohol service with a memorial/burial ground is what makes this particularly distinctive.

### 4. **No Physical Development Proposed**
While several other examples involve substantial construction, demolition, or physical modifications (the TVC studio expansion, the self-storage building, the setback amendments), this item proposes **no new permanent structures**. The entire entitlement request is about regulating *activities* rather than *built form* — making it fundamentally different in character from the development-oriented items.

### 5. **Operational Intensity on a Passive-Use Parcel**
The A1 (Agricultural) zoning and the 53-acre cemetery context suggest a quiet, low-impact use, yet the proposal would transform it into one of the more active event venues in Hollywood. The tension between the underlying zoning/land use and the proposed activity level is unique among all the examples, where the proposed uses generally align more naturally with their settings.

In summary, the unique example is an outlier because it involves **repurposing a sacred/memorial space as a high-frequency entertainment venue with alcohol service**, a combination of land use, activity type, and cultural incongruity that has no parallel among the other items.

------------------------
# Cluster: 2

## Unique Example:

2018-10-11 - 7

Proposed construction of a 136,034 square-foot, 95-unit, six-story graduate student housing development (max height 75 feet) on the USC Health Sciences Campus, replacing a surface parking lot. Requested actions include: (1) adoption of the Fourth Addendum to the previously certified USC Health Sciences Campus Project EIR; (2) a Zoning Administrator's Determination to allow shared parking between the student housing project and the adjacent USC HSC San Pablo Parking Structure; and (3) Site Plan Review for a development creating 50 or more dwelling units. The applicant is ACC OP (ALCAZAR) LP, represented by Armbruster Goldsmith & Delvac LLP.

# LLM Response:

## What Makes the Unique Example Distinctive

The unique example stands apart from the regular examples in several significant ways:

### 1. **Institutional/University Housing vs. Market-Rate or Affordable Mixed-Use Development**
The most fundamental distinction is the project's nature: this is **graduate student housing for USC** on a university health sciences campus. Every single regular example involves either market-rate residential apartments, mixed-use (residential + commercial/retail) buildings, or affordable housing developments aimed at the general public. None of the regular examples are tied to a university or institutional use.

### 2. **Absence of Affordable Housing/Density Bonus Mechanisms**
The regular examples are overwhelmingly characterized by density bonus requests, affordable housing set-asides (Very Low Income, Extremely Low Income, Workforce Housing), TOC incentives, and associated off-menu incentives and waivers. These mechanisms are a defining feature of Los Angeles residential development approvals. The unique example has **none of these**—no density bonus, no income-restricted units, no incentive requests. This makes it a stark procedural outlier.

### 3. **Shared Parking Determination Rather Than Parking Reduction Incentives**
Instead of seeking reduced parking ratios through density bonus waivers (as many regular examples do), the unique example requests a **Zoning Administrator's Determination for shared parking** between the new housing and an existing adjacent parking structure. This is a fundamentally different parking strategy—leveraging existing institutional infrastructure rather than seeking regulatory relief.

### 4. **EIR Addendum Rather Than Categorical Exemption or MND**
The environmental review involves an **addendum to a previously certified programmatic EIR** for the broader USC Health Sciences Campus. The regular examples typically rely on CEQA categorical exemptions (Class 32), Mitigated Negative Declarations, or statutory exemptions. An addendum to an existing campus-wide EIR reflects a master-planned institutional context that is absent from the other projects.

### 5. **Single-Use Project on an Institutional Campus**
The project replaces a **surface parking lot** on an existing campus—not demolishing residential or commercial structures in a neighborhood context. It is purely residential (no ground-floor retail or commercial component), which contrasts with the mixed-use character of most regular examples. The campus setting also means the project exists within a pre-planned institutional framework rather than as an independent infill development.

### 6. **Applicant Profile**
The applicant is **ACC OP (ALCAZAR) LP**, which appears to be affiliated with American Campus Communities, a specialized student housing developer—a very different entity from the typical private developers, realty partners, or investment firms seen in the regular examples.

In summary, the unique example represents **institutional student housing within a university campus ecosystem**, relying on campus-level environmental documentation and shared institutional parking, completely outside the affordable housing incentive and density bonus framework that dominates the other cases.



In [8]:
text = r"""In cluster 0, the most atypical case was an application for a conditional use permit to operate cultural events such as charitable fundraisers, film screenings, and concerts, and to sell alcoholic beverages on an existing 53-acre cemetery. 
In cluster 1, the most atypical case was an application to permit development of a 143.5 foot tall, 201,292 square foot multi-disciplinary research facility on the USC Health Sciences Campus.
In cluster 2, the most atypical case involved a Writ of Mandamus issued by the Los Angeles Superior Court ordering the City to set aside a previous approval on a 20-unit housing development because it hadn't adequately demonstrated TOC Tier 3 eligibility. However, the Writ did not limit the discretion vested in the City as to how to proceed with the project."""

with open(os.path.join(LOCAL_PATH, "results/unique_examples.tex"), 'w') as f:
    f.write(text)
